In [1]:
import os
from phreeqpython import PhreeqPython
from master_database.__main__ import main

In [2]:
pp = PhreeqPython('/home/jjl122/repos/irp-jjl122/PHREEQC_trials/PdmaDatabase1.dat')

In [3]:
sol = {
    'units': 'mol/kgw',
    'pH': 3.0,
    'Cl': 0.02,
    'Na': 0.02,
    'K': 0.02,
    'Pdma': 0.00001,
    'Zn': 0.000001,
    'temperature': 25,
    'pe': 5.92,
}
solution = pp.add_solution(sol)

 m_Zn(Pdma)-

 6.5896e-13

In [4]:
solution.species['Zn(Pdma)-']

6.589560776499402e-13

In [5]:
import pandas as pd
df = pd.read_csv('ZnPdma Speciation experiments 1-3pE+350.xls', sep='\t')
df.columns = [c.replace('m_', '').strip() for c in df.columns]


In [6]:
solution.species

{'Cl-': 0.019999973433420665,
 'H(Pdma)-2': 2.8119971666775155e-11,
 'H+': 0.001178276091931079,
 'H2': 1.0161097668699005e-21,
 'H2(Pdma)-': 5.3121910021612e-06,
 'H2O': 55.5068328911289,
 'H3(Pdma)': 4.687780218910475e-06,
 'K+': 0.020000000000000004,
 'Na+': 0.020000000000000004,
 'O2': 0.0,
 'OH-': 1.1852093526187595e-11,
 'Pdma-3': 5.960054168414869e-19,
 'Zn(Pdma)-': 6.589560776499402e-13,
 'Zn+2': 9.740271184872509e-07,
 'ZnCl+': 2.5387048037490204e-08,
 'ZnCl2': 5.75556705685394e-10,
 'ZnCl3-': 9.208245958955586e-12,
 'ZnCl4-2': 1.2784992015113148e-13,
 'ZnOHCl': 2.817434116917991e-13}

In [7]:
df_1 = pd.DataFrame(solution.species, index=[0])
df_1

,Cl-,H(Pdma)-2,H+,H2,H2(Pdma)-,H2O,H3(Pdma),K+,Na+,O2,OH-,Pdma-3,Zn(Pdma)-,Zn+2,ZnCl+,ZnCl2,ZnCl3-,ZnCl4-2,ZnOHCl
0,0.02,2.811997e-11,0.001178,1.016110e-21,0.000005,55.506833,0.000005,0.02,0.02,0.0,1.185209e-11,5.960054e-19,6.589561e-13,9.740271e-07,2.538705e-08,5.755567e-10,9.208246e-12,1.278499e-13,2.817434e-13


In [8]:
df_2 = pd.concat([df, df_1], axis=0)
df_2.dropna(axis=1, how='any', inplace=True)
pd.testing.assert_series_equal(df_2.iloc[0], df_2.iloc[1])

In [9]:
sol = {
    'units': 'mol/kgw',
    'pH': 3.0,
    'Cl': 0.1,
    'Na': 0.1,
    'As(+5)': 0.0000030,
}
pp1 = PhreeqPython('/home/jjl122/repos/irp-jjl122/databases/llnl.dat')


In [10]:
solution_1 = pp1.add_solution(sol)

In [11]:
solution_1.species['AsO4-3']

9.181771564443341e-18

9.1818e-18

AsO4-3

In [12]:
df = pd.read_csv('output.xls', sep='\t')
df.columns = [c.replace('m_', '').strip() for c in df.columns]
df.set_index('pH', inplace=True)
df.head()
keep = ['AsO4-3', 'HAsO4-2', 'H2AsO4-', 'H3AsO4']
df = df[keep]
df

,AsO4-3,HAsO4-2,H2AsO4-,H3AsO4
pH,,,,
3,9.181800e-18,9.628400e-10,2.638300e-06,3.607600e-07
4,1.022200e-15,1.074600e-08,2.948900e-06,4.034200e-08
5,1.001600e-13,1.053200e-07,2.890700e-06,3.954800e-09
6,7.618000e-12,8.010600e-07,2.198600e-06,3.007900e-10
7,2.238400e-10,2.353800e-06,6.460100e-07,8.838100e-12
8,2.774200e-09,2.917200e-06,8.006500e-08,1.095400e-13
9,2.818500e-08,2.963700e-06,8.134000e-09,1.112800e-15
10,2.605300e-07,2.738700e-06,7.515400e-10,1.028100e-17
11,1.464500e-06,1.535500e-06,4.206900e-11,5.752400e-20


In [13]:
result = pd.DataFrame()  # Initialize an empty DataFrame

for pH in df.index:
    sol = {
        'units': 'mol/kgw',
        'pH': pH,
        'Cl': 0.1,
        'Na': 0.1,
        'As(+5)': 0.0000030,
    }
    solution = pp1.add_solution(sol)
    temp_df = pd.DataFrame(solution.species, index=[pH])
    result = pd.concat([result, temp_df])

result = result[keep]
result.index.name = 'pH'
result.head()

pd.testing.assert_frame_equal(result, df)

In [14]:
import importlib.resources as pkg_resources
import master_database.clean_tables as ct

In [15]:
db = pkg_resources.files('master_database.databases').joinpath('minteq.v4.dat')
db = ct.compile_solution_species_table([db])
assert 'no_check' in db.columns
db[db['no_check']]

,equation,log_k,delta_h,gamma,d_w,v_m,millero,activity_water,add_logk,llnl_gamma,co2_llnl_gamma,erm_ddl,no_check,mole_balance,source
399,HS- = S2-2 + H+,-11.7828,"(46.4, kJ)","(0, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
400,HS- = S3-2 + H+,-10.7667,"(42.2, kJ)","(0, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
401,HS- = S4-2 + H+,-9.9608,"(39.3, kJ)","(0, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
402,HS- = S5-2 + H+,-9.3651,"(37.6, kJ)","(0, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
403,HS- = S6-2 + H+,-9.8810,"(0, kJ)","(0, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
405,Cu+ + 2HS- = Cu(S4)2-3 + 2H+,3.3900,"(0, kJ)","(23, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
406,Cu+ + 2HS- = CuS4S5-3 + 2H+,2.6600,"(0, kJ)","(25, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
407,Ag+ + 2HS- = Ag(S4)2-3 + 2H+,0.9910,"(0, kJ)","(22, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
408,Ag+ + 2HS- = AgS4S5-3 + 2H+,0.6800,"(0, kJ)","(24, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat
409,Ag+ + 2HS- = Ag(HS)S4-2 + H+,10.4310,"(0, kJ)","(15, 0)",None,None,None,None,None,None,None,None,True,None,minteq.v4.dat


In [16]:
from master_database.utils import save_master_database
save_master_database('test.dat', result_sp=db)
from master_database.parser_dat

/home/jjl122/repos/irp-jjl122/master_database/utils.py:169: UserWarning: Either solution master species or solution species is missing.
  warnings.warn("Either solution master species or solution species is missing.", UserWarning)
2024-08-20 15:47:57,529 - INFO - File processing complete.


this is working
this is working
this is working
this is working
this is working
this is working
this is working
this is working
this is working
this is working
